In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from pyproj import Transformer
import geopandas as gpd
from shapely.geometry import Point
import plotly.express as px
import branca.colormap as cm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [6]:
RAW_OPENCAFE_PATH = "../raw/포항_영업_휴게음식점.csv"
RAW_CAFE_PATH = "../raw/카페.csv"
RAW_FLOATING_PATH = "../raw/유동인구.csv"
RAW_HOSPITAL_PATH = "../raw/병원.csv"
RAW_BUSSTOP_PATH = "../raw/버스정류장.csv"
RAW_ACCOM_PATH = "../raw/숙박.csv"
RAW_BANK_PATH = "../raw/은행.csv"
RAW_SCHOOL_PATH = "../raw/학교.csv"
RAW_ACADEMY_PATH = "../raw/학원.csv"
RAW_LIBRARY_PATH = "../raw/도서관.csv"

df_opencafe = pd.read_csv(RAW_OPENCAFE_PATH, encoding="utf-8")
df_cafe = pd.read_csv(RAW_CAFE_PATH, encoding="utf-8")
df_floating = pd.read_csv(RAW_FLOATING_PATH, encoding="utf-8")
df_hospital = pd.read_csv(RAW_HOSPITAL_PATH, encoding="utf-8")
df_busstop = pd.read_csv(RAW_BUSSTOP_PATH, encoding="utf-8")
df_accom = pd.read_csv(RAW_ACCOM_PATH, encoding="utf-8")
df_bank = pd.read_csv(RAW_BANK_PATH, encoding="utf-8")
df_school = pd.read_csv(RAW_SCHOOL_PATH, encoding="utf-8")
df_academy = pd.read_csv(RAW_ACADEMY_PATH, encoding="utf-8")
df_library = pd.read_csv(RAW_LIBRARY_PATH, encoding="utf-8")

In [3]:
distance_limit = 500
cafe_period_months = 24

In [9]:
import pandas as pd
import numpy as np
from pyproj import Transformer

# --- 분석 파라미터 ---
distance_limit = 500

# 1. 카페 데이터 좌표 변환
df_target = df_opencafe.dropna(subset=['좌표정보x(epsg5174)', '좌표정보y(epsg5174)']).copy()

transformer = Transformer.from_crs("EPSG:5174", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(
    df_target['좌표정보x(epsg5174)'].values,
    df_target['좌표정보y(epsg5174)'].values
)
df_target['lon'] = lon
df_target['lat'] = lat

c_lats = df_target['lat'].values
c_lons = df_target['lon'].values

# 2. 분석 대상 인프라 목록
infra_list = [
    ('병원', df_hospital),
    ('버스정류장', df_busstop),
    ('숙박', df_accom),
    ('은행', df_bank),
    ('학교', df_school),
    ('학원', df_academy),
    ('도서관', df_library)
]

results = {}
print(f"--- 반경 {distance_limit}m 내 통합 지표 분석 시작 ---")

# 3. 각 인프라별 평균 개수 계산
for name, df_infra in infra_list:
    i_lats = df_infra['lat'].values
    i_lons = df_infra['lon'].values
    counts = [np.sum(np.sqrt(((i_lons - clon)*88000)**2 + ((i_lats - clat)*111000)**2) <= distance_limit) 
              for clat, clon in zip(c_lats, c_lons)]
    results[name] = np.mean(counts)
    print(f"✅ {name} 완료")

# 4. 경쟁 카페 수 계산
cafe_counts = [max(0, np.sum(np.sqrt(((c_lons - clon)*88000)**2 + ((c_lats - clat)*111000)**2) <= distance_limit) - 1)
               for clat, clon in zip(c_lats, c_lons)]
results['경쟁카페'] = np.mean(cafe_counts)
print(f"✅ 경쟁 카페 완료")

# 5. 평균 유동인구 계산 (nan 방지)
f_lats = df_floating['lat'].values
f_lons = df_floating['lon'].values
f_cos = df_floating['co'].values 

floating_averages = []
for clat, clon in zip(c_lats, c_lons):
    dist = np.sqrt(((f_lons - clon)*88000)**2 + ((f_lats - clat)*111000)**2)
    nearby_data = f_cos[dist <= distance_limit]
    # 데이터가 있으면 평균, 없으면 0 (nan 방지 핵심)
    floating_averages.append(np.mean(nearby_data) if len(nearby_data) > 0 else 0)

results['유동인구'] = np.mean(floating_averages)
print(f"유동인구 완료")

# --- 최종 결과 출력 (오류 수정 핵심 구간) ---
print("\n" + "="*45)
print(f"  [포항시 카페 입지 환경 평균 리포트]")
print(f"  기준 반경: {distance_limit}m / 분석 대상 카페: {len(df_target)}개")
print("="*45)

# Series로 만든 뒤 문자열 타입으로 강제 변환하여 오류 방지
summary_series = pd.Series(results).astype(object)

for idx in summary_series.index:
    val = results[idx]
    if idx == '유동인구':
        unit = "명"
        formatted_val = f"{val:,.2f}"
    else:
        unit = "개"
        formatted_val = f"{val:.2f}"

    print(f" {idx} : {formatted_val} {unit}")

print("="*45)

--- 반경 500m 내 통합 지표 분석 시작 ---
✅ 병원 완료
✅ 버스정류장 완료
✅ 숙박 완료
✅ 은행 완료
✅ 학교 완료
✅ 학원 완료
✅ 도서관 완료
✅ 경쟁 카페 완료
유동인구 완료

  [포항시 카페 입지 환경 평균 리포트]
  기준 반경: 500m / 분석 대상 카페: 1277개
 병원 : 12.92 개
 버스정류장 : 10.89 개
 숙박 : 6.84 개
 은행 : 1.05 개
 학교 : 1.21 개
 학원 : 22.44 개
 도서관 : 0.15 개
 경쟁카페 : 21.13 개
 유동인구 : 811.91 명


In [ ]:
df_opencafe = df_opencafe.dropna(subset=['인허가일자']).copy()
df_opencafe['open_date'] = pd.to_datetime(df_opencafe['인허가일자'], errors='coerce')

# 핵심 수정: 기준일을 데이터 내 가장 최신 인허가일자로 자동 설정
# (만약 유동인구에 정확히 맞추고 싶다면 reference_date = pd.to_datetime('2024-08-31') 로 변경하세요)
reference_date = df_opencafe['open_date'].max() 

# 영업 일수 계산
df_opencafe['survival_days'] = (reference_date - df_opencafe['open_date']).dt.days

# 설정한 개월 수를 일수로 환산 (1달 = 약 30.4일로 계산)
target_days = cafe_period_months * 30.4
long_survived_cafes = df_opencafe[df_opencafe['survival_days'] >= target_days].copy()

# 2. 좌표계 변환 (EPSG:5174 -> 위경도 EPSG:4326)
long_survived_cafes = long_survived_cafes.dropna(subset=['좌표정보x(epsg5174)', '좌표정보y(epsg5174)'])
transformer = Transformer.from_crs("EPSG:5174", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(
    long_survived_cafes['좌표정보x(epsg5174)'].values,
    long_survived_cafes['좌표정보y(epsg5174)'].values
)
long_survived_cafes['lon'] = lon
long_survived_cafes['lat'] = lat

print(f"기준일: {reference_date.strftime('%Y-%m-%d')}")
print(f"전체 영업 카페 {len(df_opencafe)}개 중, {cafe_period_months}개월 이상 생존한 카페는 {len(long_survived_cafes)}개 입니다.")

In [ ]:
# 3. 반경 내 병원 수 계산 (Numpy 벡터 연산)
hospital_counts = []
h_lats = df_hospital['lat'].values
h_lons = df_hospital['lon'].values

for _, cafe in long_survived_cafes.iterrows():
    c_lat = cafe['lat']
    c_lon = cafe['lon']
    
    dx = (h_lats - c_lat) * 111000
    dy = (h_lons - c_lon) * 88000
    distances = np.sqrt(dx**2 + dy**2)
    
    count = np.sum(distances <= distance_limit)
    hospital_counts.append(count)

long_survived_cafes['hospital_count'] = hospital_counts

# 4. 시각화 데이터 집계
plot_data = long_survived_cafes[long_survived_cafes['hospital_count'] > 0]
result_df = plot_data.groupby('hospital_count').size().reset_index(name='cafe_count')

# 5. 그래프 출력
plt.figure(figsize=(14, 6))
plt.bar(result_df['hospital_count'], result_df['cafe_count'], color='#EA4335', alpha=0.8)

# 제목과 라벨도 변수에 따라 자동으로 바뀌도록 f-string 적용
plt.title(f'반경 {distance_limit}m 내 병원 수에 따른 {cafe_period_months}개월 이상 생존 카페 분포', fontsize=16, pad=15)
plt.xlabel('반경 내 병원 수 (개)', fontsize=12)
plt.ylabel(f'{cafe_period_months}개월 이상 생존 카페 수 (개)', fontsize=12)

if not result_df.empty:
    plt.xticks(range(1, int(result_df['hospital_count'].max()) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 3. 반경 내 병원 수 계산 (Numpy 벡터 연산)
accom_counts = []
h_lats = df_accom['lat'].values
h_lons = df_accom['lon'].values

for _, cafe in long_survived_cafes.iterrows():
    c_lat = cafe['lat']
    c_lon = cafe['lon']
    
    dx = (h_lats - c_lat) * 111000
    dy = (h_lons - c_lon) * 88000
    distances = np.sqrt(dx**2 + dy**2)
    
    count = np.sum(distances <= distance_limit)
    accom_counts.append(count)

long_survived_cafes['accom_count'] = accom_counts

# 4. 시각화 데이터 집계
plot_data = long_survived_cafes[long_survived_cafes['accom_count'] > 0]
result_df = plot_data.groupby('accom_count').size().reset_index(name='cafe_count')

# 5. 그래프 출력
plt.figure(figsize=(14, 6))
plt.bar(result_df['accom_count'], result_df['cafe_count'], color='#EA4335', alpha=0.8)

# 제목과 라벨도 변수에 따라 자동으로 바뀌도록 f-string 적용
plt.title(f'반경 {distance_limit}m 내 병원 수에 따른 {cafe_period_months}개월 이상 생존 카페 분포', fontsize=16, pad=15)
plt.xlabel('반경 내 병원 수 (개)', fontsize=12)
plt.ylabel(f'{cafe_period_months}개월 이상 생존 카페 수 (개)', fontsize=12)

if not result_df.empty:
    plt.xticks(range(1, int(result_df['accom_count'].max()) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 3. 반경 내 병원 수 계산 (Numpy 벡터 연산)
bank_counts = []
h_lats = df_bank['lat'].values
h_lons = df_bank['lon'].values

for _, cafe in long_survived_cafes.iterrows():
    c_lat = cafe['lat']
    c_lon = cafe['lon']
    
    dx = (h_lats - c_lat) * 111000
    dy = (h_lons - c_lon) * 88000
    distances = np.sqrt(dx**2 + dy**2)
    
    count = np.sum(distances <= distance_limit)
    bank_counts.append(count)

long_survived_cafes['bank_count'] = bank_counts

# 4. 시각화 데이터 집계
plot_data = long_survived_cafes[long_survived_cafes['bank_count'] > 0]
result_df = plot_data.groupby('bank_count').size().reset_index(name='cafe_count')

# 5. 그래프 출력
plt.figure(figsize=(14, 6))
plt.bar(result_df['bank_count'], result_df['cafe_count'], color='#EA4335', alpha=0.8)

# 제목과 라벨도 변수에 따라 자동으로 바뀌도록 f-string 적용
plt.title(f'반경 {distance_limit}m 내 병원 수에 따른 {cafe_period_months}개월 이상 생존 카페 분포', fontsize=16, pad=15)
plt.xlabel('반경 내 병원 수 (개)', fontsize=12)
plt.ylabel(f'{cafe_period_months}개월 이상 생존 카페 수 (개)', fontsize=12)

if not result_df.empty:
    plt.xticks(range(1, int(result_df['bank_count'].max()) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 3. 반경 내 병원 수 계산 (Numpy 벡터 연산)
school_counts = []
h_lats = df_school['lat'].values
h_lons = df_school['lon'].values

for _, cafe in long_survived_cafes.iterrows():
    c_lat = cafe['lat']
    c_lon = cafe['lon']
    
    dx = (h_lats - c_lat) * 111000
    dy = (h_lons - c_lon) * 88000
    distances = np.sqrt(dx**2 + dy**2)
    
    count = np.sum(distances <= distance_limit)
    school_counts.append(count)

long_survived_cafes['school_count'] = school_counts

# 4. 시각화 데이터 집계
plot_data = long_survived_cafes[long_survived_cafes['school_count'] > 0]
result_df = plot_data.groupby('school_count').size().reset_index(name='cafe_count')

# 5. 그래프 출력
plt.figure(figsize=(14, 6))
plt.bar(result_df['school_count'], result_df['cafe_count'], color='#EA4335', alpha=0.8)

# 제목과 라벨도 변수에 따라 자동으로 바뀌도록 f-string 적용
plt.title(f'반경 {distance_limit}m 내 병원 수에 따른 {cafe_period_months}개월 이상 생존 카페 분포', fontsize=16, pad=15)
plt.xlabel('반경 내 병원 수 (개)', fontsize=12)
plt.ylabel(f'{cafe_period_months}개월 이상 생존 카페 수 (개)', fontsize=12)

if not result_df.empty:
    plt.xticks(range(1, int(result_df['school_count'].max()) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 3. 반경 내 병원 수 계산 (Numpy 벡터 연산)
academy_counts = []
h_lats = df_academy['lat'].values
h_lons = df_academy['lon'].values

for _, cafe in long_survived_cafes.iterrows():
    c_lat = cafe['lat']
    c_lon = cafe['lon']
    
    dx = (h_lats - c_lat) * 111000
    dy = (h_lons - c_lon) * 88000
    distances = np.sqrt(dx**2 + dy**2)
    
    count = np.sum(distances <= distance_limit)
    academy_counts.append(count)

long_survived_cafes['academy_count'] = academy_counts

# 4. 시각화 데이터 집계
plot_data = long_survived_cafes[long_survived_cafes['academy_count'] > 0]
result_df = plot_data.groupby('academy_count').size().reset_index(name='cafe_count')

# 5. 그래프 출력
plt.figure(figsize=(14, 6))
plt.bar(result_df['academy_count'], result_df['cafe_count'], color='#EA4335', alpha=0.8)

# 제목과 라벨도 변수에 따라 자동으로 바뀌도록 f-string 적용
plt.title(f'반경 {distance_limit}m 내 병원 수에 따른 {cafe_period_months}개월 이상 생존 카페 분포', fontsize=16, pad=15)
plt.xlabel('반경 내 병원 수 (개)', fontsize=12)
plt.ylabel(f'{cafe_period_months}개월 이상 생존 카페 수 (개)', fontsize=12)

if not result_df.empty:
    plt.xticks(range(1, int(result_df['academy_count'].max()) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 3. 반경 내 병원 수 계산 (Numpy 벡터 연산)
library_counts = []
h_lats = df_library['lat'].values
h_lons = df_library['lon'].values

for _, cafe in long_survived_cafes.iterrows():
    c_lat = cafe['lat']
    c_lon = cafe['lon']
    
    dx = (h_lats - c_lat) * 111000
    dy = (h_lons - c_lon) * 88000
    distances = np.sqrt(dx**2 + dy**2)
    
    count = np.sum(distances <= distance_limit)
    library_counts.append(count)

long_survived_cafes['library_count'] = library_counts

# 4. 시각화 데이터 집계
plot_data = long_survived_cafes[long_survived_cafes['library_count'] > 0]
result_df = plot_data.groupby('library_count').size().reset_index(name='cafe_count')

# 5. 그래프 출력
plt.figure(figsize=(14, 6))
plt.bar(result_df['library_count'], result_df['cafe_count'], color='#EA4335', alpha=0.8)

# 제목과 라벨도 변수에 따라 자동으로 바뀌도록 f-string 적용
plt.title(f'반경 {distance_limit}m 내 병원 수에 따른 {cafe_period_months}개월 이상 생존 카페 분포', fontsize=16, pad=15)
plt.xlabel('반경 내 병원 수 (개)', fontsize=12)
plt.ylabel(f'{cafe_period_months}개월 이상 생존 카페 수 (개)', fontsize=12)

if not result_df.empty:
    plt.xticks(range(1, int(result_df['library_count'].max()) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()